# scicp — Fine-tune Multilingual MiniLM on Scripture

Fine-tunes `paraphrase-multilingual-MiniLM-L12-v2` on **~676k** training pairs
from 7 sources across 8 languages.

**Optimized for T4 GPU — ~20–30 minutes.**

Key optimizations:
- Batch 256 via gradient accumulation (4×64) — more in-batch negatives = better MNR loss
- 2 epochs (676k pairs is enough — 3rd epoch gives diminishing returns)
- Copy data to local SSD first (Drive I/O is slow)
- Eval batch 128 (no gradients = 2× train batch fits)
- DataLoader workers=2 for parallel data loading

**Steps:**
1. Run all cells top to bottom
2. Mount Google Drive (`training-pairs.json` in `My Drive/scicp/`)
3. Training runs ~20–30 min on T4
4. Model auto-saves to Drive + zip

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q sentence-transformers datasets accelerate

In [ ]:
# ── 2. Check GPU ─────────────────────────────────────────────────────────────
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f"GPU: {gpu}  VRAM: {vram} GB")
    # Auto-select batch size based on VRAM
    if vram >= 40:    MICRO_BATCH, GRAD_ACCUM = 256, 1   # A100
    elif vram >= 22:  MICRO_BATCH, GRAD_ACCUM = 128, 2   # L4/A10
    else:             MICRO_BATCH, GRAD_ACCUM = 64, 4    # T4 (15.6 GB)
    print(f"→ micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM}, effective_batch={MICRO_BATCH * GRAD_ACCUM}")
else:
    MICRO_BATCH, GRAD_ACCUM = 32, 8
    print("No GPU — training will be very slow")

In [ ]:
# ── 3. Mount Drive + copy data to local SSD ─────────────────────────────────
# Drive I/O is slow (~30 MB/s). Local SSD is ~1.5 GB/s.
# Copying 217 MB once saves minutes of data loading during training.
import shutil, os, time
from google.colab import drive

drive.mount("/content/drive")
DRIVE_PATH = "/content/drive/MyDrive/scicp/training-pairs.json"
LOCAL_PATH = "/content/training-pairs.json"

t0 = time.time()
shutil.copy(DRIVE_PATH, LOCAL_PATH)
print(f"Copied to local SSD in {time.time()-t0:.1f}s ({os.path.getsize(LOCAL_PATH)//1024//1024} MB)")

In [ ]:
# ── 4. Load + prepare dataset ────────────────────────────────────────────────
import json, random
from datasets import Dataset

t0 = time.time()
with open(LOCAL_PATH) as f:
    pairs = json.load(f)
print(f"Loaded {len(pairs):,} pairs in {time.time()-t0:.1f}s")

random.seed(42)
random.shuffle(pairs)

# 97/3 split — with 676k pairs, 3% validation (~20k) is more than enough
split = int(len(pairs) * 0.97)
train_ds = Dataset.from_dict({
    "anchor":   [p["anchor"]   for p in pairs[:split]],
    "positive": [p["positive"] for p in pairs[:split]],
})
val_ds = Dataset.from_dict({
    "anchor":   [p["anchor"]   for p in pairs[split:]],
    "positive": [p["positive"] for p in pairs[split:]],
})
del pairs  # free ~400 MB RAM

print(f"train={len(train_ds):,}  val={len(val_ds):,}")
print("Sample:", train_ds[0])

In [ ]:
# ── 5. Fine-tune ─────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer

BASE_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
OUT_DIR    = "/content/scripture-minilm"
EPOCHS     = 2       # 676k × 2 = 1.35M samples; diminishing returns after epoch 2

effective_batch = MICRO_BATCH * GRAD_ACCUM  # 256 on T4
total_steps     = (len(train_ds) // effective_batch) * EPOCHS
warmup_steps    = total_steps // 20  # 5% warmup

print(f"Effective batch: {effective_batch}")
print(f"Steps/epoch: {len(train_ds) // effective_batch:,}")
print(f"Total steps: {total_steps:,}  Warmup: {warmup_steps}")
print(f"In-batch negatives per sample: {effective_batch - 1}")

model = SentenceTransformer(BASE_MODEL)
loss  = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=MICRO_BATCH,
    per_device_eval_batch_size=MICRO_BATCH * 2,  # no gradients = 2x fits
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=warmup_steps,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,          # keep only best + last (saves disk)
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    bf16=False,
    dataloader_num_workers=2,    # parallel data loading
    dataloader_pin_memory=True,  # faster CPU to GPU transfer
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=loss,
)

t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"
✅ Done in {elapsed:.1f} min  ({len(train_ds) * EPOCHS / elapsed:.0f} pairs/min)")

In [ ]:
# ── 6. Save model to Google Drive ───────────────────────────────────────────
import os, shutil

DRIVE_OUT = "/content/drive/MyDrive/scicp/scripture-minilm"
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copytree(OUT_DIR, DRIVE_OUT, dirs_exist_ok=True)
print(f"Model saved to Drive: {DRIVE_OUT}")
print("Files:", [f for f in os.listdir(DRIVE_OUT) if not f.startswith("checkpoint")])

# Zip for easy download
shutil.make_archive("/content/scripture-minilm", "zip", OUT_DIR)
ZIP_PATH = "/content/drive/MyDrive/scicp/scripture-minilm.zip"
shutil.copy("/content/scripture-minilm.zip", ZIP_PATH)
print(f"Zip: {os.path.getsize(ZIP_PATH) // 1024 // 1024} MB")

## After training

The fine-tuned model is saved to **My Drive → scicp/scripture-minilm/**.

On your local machine:

```bash
# 1. Download the model folder from Google Drive to:
#    resources/models/scripture-minilm/

# 2. Re-encode all 41k verses with the fine-tuned model (~5 min)
python3 scripts/rebake-embeddings.py

# 3. Re-whiten embeddings (ZCA whitening matrix has changed)
node scripts/prebake-whitening.js

# 4. Rebuild cluster labels (centroids have changed)
node scripts/prebake-cluster-labels.js

# 5. Rebuild kNN graph
node scripts/prebake-knn.js

# 6. Rebuild spectral embeddings
node scripts/prebake-spectral.js

# 7. Restart the server
npm run dev
```

### Optimization summary

| | Old notebook | This notebook |
|---|---|---|
| Base model | `all-MiniLM-L6-v2` (English, 6L) | `paraphrase-multilingual-MiniLM-L12-v2` (50+ langs, 12L) |
| Training pairs | ~62k (topical guide) | ~676k (7 sources × 8 languages) |
| Effective batch | 128 | 256 (64×4 grad accum) — 255 in-batch negatives |
| Epochs | 4 | 2 (10× more data = fewer passes needed) |
| Data I/O | Read from Drive (slow) | Copy to local SSD first |
| Eval batch | Same as train | 2× train (no gradients) |
| DataLoader | Default | workers=2, pin_memory=True |
| GPU auto-detect | No | Yes (A100/L4/T4 auto-config) |
| Val split | 10% (~6k) | 3% (~20k) — still large enough, more for training |
| Est. time (T4) | ~15–20 min | ~20–30 min (11× data in only 1.5× time) |